# Step-by-step demo of NEDAS using the vort2d model


## Install NEDAS and dependencies
NEDAS is available from the PyPI platform. You can install with ```pip install NEDAS```.

You can also clone the `NEDAS` repository and install in editable mode ```cd NEDAS; pip install -e .``` for active code development.

In [ ]:
# 1. Install the latest version of NEDAS on develop branch
%cd ~
%rm -rf NEDAS
!git clone https://github.com/nansencenter/NEDAS.git
%cd NEDAS
%pip install .

# 2. Install additional dependencies
# numba provides JIT compilation of python function to speed it up during runtime
%pip install numba
# cmocean provides better colormaps for visualization
%pip install cmocean

# 3. If in Google Colab, need to clone the tutorial repo too
import sys
if 'google.colab' in sys.modules:
    %cd ~
    !git clone https://github.com/myying/NEDAS_tutorials.git
    %cd NEDAS_tutorials

In [ ]:
# Load utility modules
# numpy
import numpy as np
# a few graphic utility modules
import matplotlib.pyplot as plt
import cmocean
from matplotlib import cm
from matplotlib.animation import FuncAnimation
from IPython.core.display import HTML


## Initialize configuration and main scheme

In [ ]:
# load configuration YAML file
from NEDAS.config import Config
config = Config(config_file="vort2d/config.yml")

In [ ]:
# initialize the main Scheme
from NEDAS import get_scheme
scheme = get_scheme(config)

In [ ]:
# the model object 
model = scheme.c.models['vort2d']

from NEDAS.models.vort2d import Vort2DModel
assert isinstance(model, Vort2DModel)

In [ ]:
# 
dataset = scheme.c.datasets['vort2d']

from NEDAS.datasets.vort2d import Vort2DObs
assert isinstance(dataset, Vort2DObs)

In [ ]:
from NEDAS.grid import RegularGrid
grid = scheme.c.grid

## Prepare truth

check nc files in vort2d/truth

In [ ]:
scheme.c.time = config.time_start

In [ ]:
%rm -rf vort2d/work/truth

In [ ]:
model.loc_sprd = 0
scheme.run_step('prepare_truth')

## Prepare initial ensemble and run forecasts

In [ ]:
%rm -rf vort2d/work/init_ens

In [ ]:
model.loc_sprd = 100000
scheme.run_step('prepare_init_ensemble')

In [ ]:
%rm -rf vort2d/work/cycle

In [ ]:
scheme.c.progress.call_stack = []
scheme.c.progress.push('')
scheme.c.time = config.time_start
scheme.c.log_event("CYCLING START...")
while scheme.c.time < config.time_end:
    scheme.c.log_event(f"CURRENT CYCLE: {scheme.c.time} -> {scheme.c.next_time}")
    scheme.run_step('preprocess')
    scheme.run_step('ensemble_forecast')
    scheme.c.time = scheme.c.next_time
scheme.c.log_event("CYCLING COMPLETE.", flag='finish')

In [ ]:
scheme.c.time = config.time_start
truth_state = []
times = []
while scheme.c.time < config.time_end:
    times.append(scheme.c.time)

    truth = model.read_var(path = model.truth_dir, name='velocity', member=None, time=scheme.c.time)
    truth_state.append(truth)
    scheme.c.time = scheme.c.next_time

In [ ]:
scheme.c.time = config.time_start
forecast_state = []
for n in range(10):
    path = scheme.c.fs.forecast_dir(scheme.c.time, 'vort2d')
    ens = []
    for m in range(scheme.c.nens):
        ens.append(model.read_var(path=path, name='velocity', member=m, time=scheme.c.time))
    forecast_state.append(ens)
    scheme.c.time = scheme.c.next_time

In [ ]:
from NEDAS.utils.spatial_operation import gradx, grady

def to_vorticity(vel):
    u, v = vel[0], vel[1]
    zeta = gradx(v, grid.dx, grid.cyclic_dim) - grady(u, grid.dy, grid.cyclic_dim)
    return zeta

fig, ax = plt.subplots(1, 3, figsize=(9, 3))
nt = 10 #len(truth_state)
clvs = [1e-3]
current_contour_sets = []

def update(n):
    global current_contour_sets
    for cs in current_contour_sets:
        cs.remove() 
    current_contour_sets.clear()

    pc_truth = ax[0].pcolor(to_vorticity(truth_state[n]), vmin=-5e-3, vmax=5e-3, cmap='bwr')
    current_contour_sets.append(pc_truth)
    cmap = [plt.cm.jet(x) for x in np.linspace(0, 1, scheme.c.nens)]
    for m in range(scheme.c.nens):
        cs_ens = ax[0].contour(to_vorticity(forecast_state[n][m]), clvs, colors=[cmap[m][0:3]], linewidths=0.5)
        current_contour_sets.append(cs_ens)
    cs_truth = ax[0].contour(to_vorticity(truth_state[n]), clvs, colors='k', linewidths=1.5)
    current_contour_sets.append(cs_truth)
    ax[0].set_title(f"Forecast {n*config.cycle_period} h")
    return current_contour_sets

ani = FuncAnimation(fig, update, frames=range(nt), blit=False, interval=100)
plt.close()
HTML(ani.to_jshtml())

## Substeps in an analysis cycle

In [ ]:
scheme.c.time = config.time_start

## Running the entire scheme

Cycling from time_start to time_end

In [ ]:
scheme.c.time = config.time_start
scheme.c.progress.call_stack = []

scheme()


In [ ]:
scheme.c.time = config.time_start
prior_state = []
post_state = []
while scheme.c.time < config.time_end:

    path = scheme.c.fs.forecast_dir(scheme.c.prev_time, 'vort2d')
    ens = []
    for m in range(scheme.c.nens):
        ens.append(model.read_var(path=path, name='velocity', member=m, time=scheme.c.time))
    prior_state.append(ens)

    ens = []
    for m in range(scheme.c.nens):
        if config.run_analysis and scheme.c.time >= config.time_analysis_start and scheme.c.time <= config.time_analysis_end:
            path = scheme.c.fs.forecast_dir(scheme.c.time, 'vort2d')
            ens.append(model.read_var(path=path, name='velocity', member=m, time=scheme.c.time))
        else:
            ens.append(np.full(truth.shape, np.nan))
    post_state.append(ens)

    scheme.c.time = scheme.c.next_time

In [ ]:
from NEDAS.utils.spatial_operation import gradx, grady

def to_vorticity(vel):
    u, v = vel[0], vel[1]
    zeta = gradx(v, grid.dx, grid.cyclic_dim) - grady(u, grid.dy, grid.cyclic_dim)
    return zeta

fig, ax = plt.subplots(1, 3, figsize=(9, 3))
nt = 10 #len(truth_state)
clvs = [1e-3]
current_contour_sets = []

def update(n):
    global current_contour_sets
    for cs in current_contour_sets:
        cs.remove() 
    current_contour_sets.clear()

    pc_truth = ax[0].pcolormesh(to_vorticity(truth_state[n]), vmin=-5e-3, vmax=5e-3, cmap='bwr')
    current_contour_sets.append(pc_truth)
    cmap = [plt.cm.jet(x) for x in np.linspace(0, 1, scheme.c.nens)]
    for m in range(scheme.c.nens):
        cs_ens = ax[0].contour(to_vorticity(prior_state[n][m]), clvs, colors=[cmap[m][0:3]], linewidths=0.5)
        current_contour_sets.append(cs_ens)
    cs_truth = ax[0].contour(to_vorticity(truth_state[n]), clvs, colors='k', linewidths=1.5)
    current_contour_sets.append(cs_truth)
    ax[0].set_title(f"Forecast {n*config.cycle_period} h")
    return current_contour_sets

ani = FuncAnimation(fig, update, frames=range(nt), blit=False, interval=100)
plt.close()
HTML(ani.to_jshtml())

## A few things to try to play with
